<a href="https://colab.research.google.com/github/Maee127/Adversarial-Notebooks/blob/master/%239/Notebook-09.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# Adversarial Attacks Series — Note 09
# Confidently Wrong: The Human Cost of Silent Failure
# =============================================================
#
# Series:  Humble Model / The Human Cost of Silent Failure
# Dataset: CIFAR-10
# Model:   Deep Ensembles (loaded from Essay #7 checkpoints)
#
# CHANGES IN THIS REVISION:
#   1. SIGN BUG FIXED in pgd_targeted_gate. The previous version
#      negated cls_loss (turning "ascend to maximize error" into
#      "ascend to minimize error") and left agreement_loss
#      unnegated (turning "ascend to suppress disagreement" into
#      "ascend to increase disagreement"). Both were backwards.
#      As written, the attack would have made the ensemble more
#      accurate AND more likely to defer -- the opposite of its
#      purpose, and the kind of bug that produces a plausible-
#      looking but wrong result rather than an obvious crash.
#   2. LAMBDA SWEEP added (Part F.5). A single hardcoded lambda
#      risks exactly the "obfuscated gradients give a false sense
#      of security" failure this essay cites -- an untuned attack
#      hyperparameter can make a defense look stronger than it is.
#      The sweep runs on a small subset and selects the lambda that
#      maximizes wrong-and-undeferred predictions (the human-cost
#      metric), then that lambda is used for the full evaluation.
#   3. DIAGNOSTIC DRY RUN added (Part F.6). Before committing to
#      the full 8,000-image evaluation (40 steps x 5 models x 8,000
#      images is expensive), this runs the attack on a small batch
#      and prints cls_loss and agreement_loss at the first and last
#      step, so a sign error is caught in minutes rather than
#      discovered after a full run produces a misleadingly clean
#      table.
#
# Notebook structure:
#   Part A: Imports and Setup
#   Part B: Dataset Loading (CIFAR-10, held-out split)
#   Part C: Load Deep Ensembles from Essay #7
#   Part D: Reconstruct the Calibrated Gate
#   Part E: Ordinary Transferred Attack (baseline PGD)
#   Part F: Targeted Gate Attack (misclassify + preserve agreement)
#   Part F.5: Lambda Sweep
#   Part F.6: Diagnostic Dry Run
#   Part G: Evaluation — the Gap
#   Part H: Head-to-Head with Essay #7
#   Part I: Summary and Outputs
# =============================================================


In [ ]:
# -------------------------------------------------------------
# Part A: Imports and Setup
# -------------------------------------------------------------

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from tqdm import tqdm
import random
import json

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


Using device: cuda


In [ ]:
# -------------------------------------------------------------
# Part B: Dataset Loading (CIFAR-10) — same held-out split as #6-#8
# -------------------------------------------------------------

cifar_transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

cifar_transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=cifar_transform_train)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform_test)

# SAME seed and split as Essays #6–#8 so calibration/evaluation match
calib_size = int(0.2 * len(test_dataset))
eval_size  = len(test_dataset) - calib_size
calib_dataset, eval_dataset = random_split(
    test_dataset, [calib_size, eval_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
calib_loader = DataLoader(calib_dataset, batch_size=64, shuffle=False, num_workers=0)
eval_loader  = DataLoader(eval_dataset,  batch_size=64, shuffle=False, num_workers=0)

print(f"Training samples:    {len(train_dataset):,}")
print(f"Calibration samples: {len(calib_dataset):,}")
print(f"Evaluation samples:  {len(eval_dataset):,}")

# Per-channel valid normalized range
CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
CIFAR_STD  = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
CIFAR_MIN  = ((0 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)
CIFAR_MAX  = ((1 - CIFAR_MEAN) / CIFAR_STD).to(DEVICE)

def clamp_valid(x):
    return torch.max(torch.min(x, CIFAR_MAX), CIFAR_MIN)


100%|██████████| 170M/170M [08:50<00:00, 321kB/s]


Training samples:    50,000
Calibration samples: 2,000
Evaluation samples:  8,000


In [ ]:
from google.colab import files

files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("essay7_ensemble_checkpoints.zip", "r") as archive:
    archive.extractall("/content")

print("Checkpoints found:")
!ls -lh /content/checkpoint_ensemble_*.pth

In [ ]:
# -------------------------------------------------------------
# Part C: Load Deep Ensembles from Essay #7
# -------------------------------------------------------------
#
# Essay #7 trained five independent CNN members and saved each as
# checkpoint_ensemble_*.pth. We reload them exactly as they were.
# -------------------------------------------------------------

class BackboneCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.fc1   = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x)); x = F.max_pool2d(x, 2)
        x = F.relu(self.conv3(x)); x = F.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return x

class StandardCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = BackboneCNN()
        self.fc_out   = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.fc_out(self.backbone(x))

NUM_ENSEMBLE = 5
ensemble_models = []

for i in range(NUM_ENSEMBLE):
    ckpt_path = f'checkpoint_ensemble_{i}.pth'
    assert os.path.exists(ckpt_path), (
        f"Missing {ckpt_path}. Run Essay #7's notebook first, "
        f"or place the ensemble checkpoints in the working directory."
    )
    model = StandardCNN().to(DEVICE)
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    ensemble_models.append(model)
    print(f"Loaded ensemble member {i}")

# Sanity: ensemble clean accuracy on the evaluation split
def ensemble_predict(models, images):
    """Return mean_probs [B, C] and max_variance [B] across members."""
    probs = torch.stack(
        [torch.softmax(m(images), dim=1) for m in models], dim=0
    )                                       # [M, B, C]
    mean_probs  = probs.mean(dim=0)         # [B, C]
    variance    = probs.var(dim=0)          # [B, C]
    max_variance = variance.max(dim=1).values  # [B]
    return mean_probs, max_variance

def evaluate_accuracy_ensemble(models, loader):
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            mean_probs, _ = ensemble_predict(models, images)
        preds = mean_probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return 100 * correct / total

clean_acc = evaluate_accuracy_ensemble(ensemble_models, eval_loader)
print(f"\nDeep Ensemble clean accuracy (eval split): {clean_acc:.2f}%")


Loaded ensemble member 0
Loaded ensemble member 1
Loaded ensemble member 2
Loaded ensemble member 3
Loaded ensemble member 4

Deep Ensemble clean accuracy (eval split): 84.90%


In [ ]:
# -------------------------------------------------------------
# Part D: Reconstruct the Calibrated Gate
# -------------------------------------------------------------
#
# Essay #7 calibrated the ensemble's deferral threshold at 25% clean
# deferral on the calibration split. We rebuild exactly that.
# -------------------------------------------------------------

def collect_uncertainty(models, loader):
    """Collect mean predictions, max-variance scores, and labels."""
    scores_all, preds_all, labels_all = [], [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        with torch.no_grad():
            mean_probs, max_variance = ensemble_predict(models, images)
        scores_all.append(max_variance.cpu())
        preds_all.append(mean_probs.argmax(dim=1).cpu())
        labels_all.append(labels)
    return (
        torch.cat(scores_all).numpy(),
        torch.cat(preds_all).numpy(),
        torch.cat(labels_all).numpy(),
    )

print("\nCalibrating gate on the clean calibration split...")
cal_scores, cal_preds, cal_labels = collect_uncertainty(ensemble_models, calib_loader)

TARGET_CLEAN_DEFERRAL = 0.25
THRESHOLD = float(np.quantile(cal_scores, 1.0 - TARGET_CLEAN_DEFERRAL))

cal_deferral = float((cal_scores > THRESHOLD).mean())
print(f"  Calibrated threshold: {THRESHOLD:.6f}")
print(f"  Calibration deferral: {cal_deferral*100:.2f}%")



Calibrating gate on the clean calibration split...
  Calibrated threshold: 0.054999
  Calibration deferral: 25.00%


In [ ]:

# -------------------------------------------------------------
# Part E: Ordinary Transferred Attack (baseline PGD)
# -------------------------------------------------------------
#
# Same as previous essays: PGD against a baseline CNN, transferred.
# We do NOT have Essay #7's baseline checkpoint here, so we train a
# small baseline for the attack. NOTE: this baseline is trained
# fresh for this notebook and is NOT guaranteed to reproduce
# Essay #7's exact transferred-attack numbers -- check the
# "transferred" row against Essay #7's reported 15.42% in Part H
# before treating that comparison as controlled. If it diverges
# substantially, the baseline/attack setup differs from #7's, and
# only the WITHIN-ESSAY gap (targeted vs. this essay's own
# transferred attack) should be trusted, not a direct read against
# #7's number.
# -------------------------------------------------------------

def train_baseline(model, loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        for images, labels in tqdm(loader, desc=f"Baseline epoch {epoch}"):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
    return model

baseline_model = StandardCNN().to(DEVICE)
if os.path.exists('checkpoint_baseline_essay9.pth'):
    baseline_model.load_state_dict(
        torch.load('checkpoint_baseline_essay9.pth', map_location=DEVICE)['model_state_dict']
    )
    print("Loaded baseline (for attack) from checkpoint")
else:
    print("Training baseline CNN for the transferred attack...")
    baseline_model = train_baseline(baseline_model, train_loader, epochs=10)
    torch.save({'model_state_dict': baseline_model.state_dict()},
               'checkpoint_baseline_essay9.pth')
    print("Saved baseline checkpoint")

baseline_model.eval()

def pgd_transferred(attack_model, images, labels,
                    epsilon=0.03, step_size=0.007, num_steps=40):
    """Standard PGD on attack_model; result is transferred to the ensemble."""
    attack_model.eval()
    original = images.clone().detach().to(DEVICE)
    labels   = labels.to(DEVICE)
    adv      = original.clone().detach()

    for _ in range(num_steps):
        adv.requires_grad_(True)
        loss = F.cross_entropy(attack_model(adv), labels)
        grad = torch.autograd.grad(loss, adv)[0]
        updated = adv.detach() + step_size * grad.sign()
        perturbation = torch.clamp(updated - original, -epsilon, epsilon)
        adv = clamp_valid(original + perturbation).detach()
    return adv


Training baseline CNN for the transferred attack...


Baseline epoch 9: 100%|██████████| 782/782 [00:21<00:00, 36.07it/s]

Saved baseline checkpoint


In [ ]:

# -------------------------------------------------------------
# Part F: Targeted Gate Attack — SIGN BUG FIXED
# -------------------------------------------------------------
#
# Objective:
#   1. Push the ensemble's mean prediction toward wrong classes
#   2. Keep the members in agreement (so max-variance stays below
#      the calibrated deferral threshold)
#
# Combined loss (ascended via PGD gradient ascent):
#   L = L_cls  -  lambda_agreement * L_agreement
#
# where L_cls = cross-entropy against the true label. Ascending it
# directly increases error -- no negation needed, since the loop
# already ascends. (The earlier version negated this, which meant
# ascending it actually DECREASED error -- backwards.)
#
# L_agreement = mean max-variance across ensemble members.
# Ascending (L_cls - lambda*L_agreement) requires L_agreement to go
# DOWN for the combined objective to go up -- i.e. the attack is
# rewarded for suppressing disagreement, not increasing it. (The
# earlier version added this term unsubtracted, which rewarded
# INCREASING disagreement -- also backwards.)
# -------------------------------------------------------------

def pgd_adaptive_gate(models, images, labels, threshold,
                      epsilon=0.03, step_size=0.007, num_steps=40,
                      lambda_agreement=1.0, return_trace=False):
    original = images.clone().detach().to(DEVICE)
    labels   = labels.to(DEVICE)
    adv      = original.clone().detach()

    trace = {'cls_loss': [], 'agreement_loss': []} if return_trace else None

    for step in range(num_steps):
        adv.requires_grad_(True)

        # Forward pass through all members
        logits = torch.stack([m(adv) for m in models], dim=0)   # [M, B, C]
        probs  = torch.softmax(logits, dim=2)                    # [M, B, C]

        # 1) Classification loss: cross-entropy against the true label.
        #    Ascending this directly increases error. No negation.
        mean_probs = probs.mean(dim=0)                           # [B, C]
        cls_loss = F.nll_loss(torch.log(mean_probs + 1e-10), labels)

        # 2) Agreement loss: mean max-variance across members.
        #    We want this to go DOWN as the combined objective goes UP,
        #    so it enters the combined loss with a minus sign.
        variance = probs.var(dim=0).max(dim=1).values            # [B]
        agreement_loss = variance.mean()

        # Combined objective -- ascend to increase error AND suppress variance
        loss = cls_loss - lambda_agreement * agreement_loss

        if return_trace:
            trace['cls_loss'].append(cls_loss.item())
            trace['agreement_loss'].append(agreement_loss.item())

        grad = torch.autograd.grad(loss, adv)[0]

        updated = adv.detach() + step_size * grad.sign()
        perturbation = torch.clamp(updated - original, -epsilon, epsilon)
        adv = clamp_valid(original + perturbation).detach()

    if return_trace:
        return adv, trace
    return adv


In [ ]:

# -------------------------------------------------------------
# Part F.5: Lambda Sweep
# -------------------------------------------------------------
#
# A single hardcoded lambda risks exactly the "obfuscated gradients
# give a false sense of security" failure mode (Athalye, Carlini &
# Wagner, 2018) that this essay cites. Sweep on a small subset of
# the evaluation split, and select the lambda that does the most
# damage -- i.e. maximizes wrong-and-undeferred predictions, the
# human-cost metric this essay is actually about. That lambda is
# then used for the full evaluation in Part G.
# -------------------------------------------------------------

def quick_eval_adaptive(models, threshold, loader, lambda_agreement,
                        attack_params, max_samples=500):
    """Lightweight version of the Part G evaluator, for sweeping lambda
    on a subset rather than the full 8,000-image set."""
    n_seen = 0
    n_def = n_pred = n_correct_pred = n_wrong_undeferred = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv = pgd_adaptive_gate(models, images, labels, threshold,
                                lambda_agreement=lambda_agreement, **attack_params)

        with torch.no_grad():
            mean_probs, max_variance = ensemble_predict(models, adv)
        preds = mean_probs.argmax(dim=1)
        defer = max_variance > threshold
        correct = (preds == labels)

        total += labels.size(0)
        n_def += int(defer.sum().item())
        n_pred += int((~defer).sum().item())
        n_correct_pred += int((correct & ~defer).sum().item())
        n_wrong_undeferred += int(((~correct) & (~defer)).sum().item())

        n_seen += labels.size(0)
        if n_seen >= max_samples:
            break

    acc_on_pred = 100.0 * n_correct_pred / n_pred if n_pred else 0.0
    return {
        'wrong_and_undeferred_pct': 100.0 * n_wrong_undeferred / total if total else 0.0,
        'risk_on_predicted': 100.0 - acc_on_pred,
        'deferral_rate': 100.0 * n_def / total if total else 0.0,
    }

LAMBDA_GRID = [0.1, 0.5, 1.0, 2.0, 5.0]
SWEEP_PARAMS = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}
SWEEP_SAMPLES = 500

print("\n" + "=" * 70)
print(f"Lambda sweep ({SWEEP_SAMPLES}-image subset, grid = {LAMBDA_GRID})")
print("=" * 70)

sweep_results = {}
for lam in LAMBDA_GRID:
    print(f"\nTesting lambda_agreement = {lam} ...")
    result = quick_eval_adaptive(ensemble_models, THRESHOLD, eval_loader,
                                  lambda_agreement=lam, attack_params=SWEEP_PARAMS,
                                  max_samples=SWEEP_SAMPLES)
    sweep_results[lam] = result
    print(f"  wrong & undeferred: {result['wrong_and_undeferred_pct']:.2f}%   "
          f"risk on predicted: {result['risk_on_predicted']:.2f}%   "
          f"deferral: {result['deferral_rate']:.2f}%")

BEST_LAMBDA = max(sweep_results, key=lambda l: sweep_results[l]['wrong_and_undeferred_pct'])
print(f"\nSelected lambda_agreement = {BEST_LAMBDA} "
      f"(maximizes wrong-and-undeferred at {sweep_results[BEST_LAMBDA]['wrong_and_undeferred_pct']:.2f}% "
      f"on the {SWEEP_SAMPLES}-image subset)")
print("This is the lambda used for the full evaluation below.")



Lambda sweep (500-image subset, grid = [0.1, 0.5, 1.0, 2.0, 5.0])

Testing lambda_agreement = 0.1 ...
  wrong & undeferred: 22.85%   risk on predicted: 59.39%   deferral: 61.52%

Testing lambda_agreement = 0.5 ...
  wrong & undeferred: 24.80%   risk on predicted: 59.07%   deferral: 58.01%

Testing lambda_agreement = 1.0 ...
  wrong & undeferred: 26.95%   risk on predicted: 50.92%   deferral: 47.07%

Testing lambda_agreement = 2.0 ...
  wrong & undeferred: 33.59%   risk on predicted: 45.99%   deferral: 26.95%

Testing lambda_agreement = 5.0 ...
  wrong & undeferred: 42.97%   risk on predicted: 45.36%   deferral: 5.27%

Selected lambda_agreement = 5.0 (maximizes wrong-and-undeferred at 42.97% on the 500-image subset)
This is the lambda used for the full evaluation below.


In [ ]:

# -------------------------------------------------------------
# Part F.6: Diagnostic Dry Run
# -------------------------------------------------------------
#
# Before the full 8,000-image run, confirm the attack is actually
# moving in the right direction: cross-entropy should INCREASE over
# the 40 steps, and the agreement/variance term should DECREASE.
# If either trend is flat or backwards, stop here -- something is
# still wrong and a full run would just waste compute reproducing
# the same mistake at scale.
# -------------------------------------------------------------

print("\n" + "=" * 70)
print("Diagnostic dry run (small batch, selected lambda)")
print("=" * 70)

dry_images, dry_labels = next(iter(eval_loader))
dry_images, dry_labels = dry_images.to(DEVICE), dry_labels.to(DEVICE)

_, trace = pgd_adaptive_gate(
    ensemble_models, dry_images, dry_labels, THRESHOLD,
    lambda_agreement=BEST_LAMBDA, return_trace=True, **SWEEP_PARAMS
)

print(f"\ncls_loss (cross-entropy, should RISE):")
print(f"  step 0  : {trace['cls_loss'][0]:.4f}")
print(f"  step {len(trace['cls_loss'])-1} : {trace['cls_loss'][-1]:.4f}")

print(f"\nagreement_loss (ensemble variance, should FALL):")
print(f"  step 0  : {trace['agreement_loss'][0]:.6f}")
print(f"  step {len(trace['agreement_loss'])-1} : {trace['agreement_loss'][-1]:.6f}")

cls_rising = trace['cls_loss'][-1] > trace['cls_loss'][0]
var_falling = trace['agreement_loss'][-1] < trace['agreement_loss'][0]

if cls_rising and var_falling:
    print("\n  OK -- both trends are in the expected direction. Safe to proceed to the full run.")
else:
    print("\n  WARNING -- at least one trend is NOT in the expected direction.")
    print("  Do not proceed to the full evaluation until this is resolved.")
    if not cls_rising:
        print("  -> cls_loss did not rise: the attack is not increasing error.")
    if not var_falling:
        print("  -> agreement_loss did not fall: the attack is not suppressing disagreement.")



Diagnostic dry run (small batch, selected lambda)

cls_loss (cross-entropy, should RISE):
  step 0  : 0.5389
  step 39 : 1.7223

agreement_loss (ensemble variance, should FALL):
  step 0  : 0.038669
  step 39 : 0.015358

  OK -- both trends are in the expected direction. Safe to proceed to the full run.


In [ ]:

# -------------------------------------------------------------
# Part G: Evaluation — The Gap
# -------------------------------------------------------------

def evaluate_under_attack(models, threshold, loader, attack='none',
                          attack_params=None, max_samples=None):
    """
    Evaluate ensemble + gate under a specific attack.
    Reports: coverage, deferral_rate, risk_on_predicted,
    AND wrong_and_undeferred (the human-cost number).
    """
    n_seen = 0
    n_pred = n_def = 0
    n_correct_pred = 0
    n_wrong_undeferred = 0   # explicitly tracked — this is the human cost
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if attack == 'transferred':
            images = pgd_transferred(baseline_model, images, labels, **attack_params)
        elif attack == 'targeted':
            images = pgd_adaptive_gate(models, images, labels, threshold,
                                       **attack_params)

        with torch.no_grad():
            mean_probs, max_variance = ensemble_predict(models, images)
        preds  = mean_probs.argmax(dim=1)
        defer  = max_variance > threshold
        correct = (preds == labels)

        total += labels.size(0)
        n_def += int(defer.sum().item())
        n_pred += int((~defer).sum().item())
        n_correct_pred += int((correct & ~defer).sum().item())
        # Wrong AND not deferred — the failure the human sees
        n_wrong_undeferred += int(((~correct) & (~defer)).sum().item())

        if max_samples is not None:
            n_seen += labels.size(0)
            if n_seen >= max_samples:
                break

    coverage = 100.0 * n_pred / total if total else 0.0
    deferral = 100.0 * n_def / total if total else 0.0
    acc_on_pred = 100.0 * n_correct_pred / n_pred if n_pred else 0.0
    risk_on_pred = 100.0 - acc_on_pred
    wrong_undeferred_pct = 100.0 * n_wrong_undeferred / total if total else 0.0

    return {
        'total': total,
        'coverage': coverage,
        'deferral_rate': deferral,
        'accuracy_on_predicted': acc_on_pred,
        'risk_on_predicted': risk_on_pred,
        'wrong_and_undeferred_count': n_wrong_undeferred,
        'wrong_and_undeferred_pct': wrong_undeferred_pct,
    }

print("\n" + "=" * 70)
print("Evaluation of Deep Ensembles under three conditions")
print("=" * 70)

# Clean
clean_metrics = evaluate_under_attack(ensemble_models, THRESHOLD, eval_loader,
                                       attack='none')
print("\nCLEAN:")
print(f"  coverage           : {clean_metrics['coverage']:.2f}%")
print(f"  deferral rate      : {clean_metrics['deferral_rate']:.2f}%")
print(f"  risk on predicted  : {clean_metrics['risk_on_predicted']:.2f}%")
print(f"  wrong & undeferred : {clean_metrics['wrong_and_undeferred_pct']:.2f}%")

# Ordinary transferred attack
transferred_params = {'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40}
transferred_metrics = evaluate_under_attack(
    ensemble_models, THRESHOLD, eval_loader,
    attack='transferred', attack_params=transferred_params
)
print("\nTRANSFERRED PGD (baseline-attacked, transferred to ensemble):")
print(f"  coverage           : {transferred_metrics['coverage']:.2f}%")
print(f"  deferral rate      : {transferred_metrics['deferral_rate']:.2f}%")
print(f"  risk on predicted  : {transferred_metrics['risk_on_predicted']:.2f}%")
print(f"  wrong & undeferred : {transferred_metrics['wrong_and_undeferred_pct']:.2f}%")

# Targeted gate attack — using the lambda selected by the sweep
targeted_params = {
    'epsilon': 0.03, 'step_size': 0.007, 'num_steps': 40,
    'lambda_agreement': BEST_LAMBDA,
}
print(f"\nRunning targeted gate attack with lambda_agreement = {BEST_LAMBDA} "
      f"(slower — gradients through all members)...")
targeted_metrics = evaluate_under_attack(
    ensemble_models, THRESHOLD, eval_loader,
    attack='targeted', attack_params=targeted_params
)
print("\nTARGETED GATE ATTACK (misclassify + preserve agreement):")
print(f"  coverage           : {targeted_metrics['coverage']:.2f}%")
print(f"  deferral rate      : {targeted_metrics['deferral_rate']:.2f}%")
print(f"  risk on predicted  : {targeted_metrics['risk_on_predicted']:.2f}%")
print(f"  wrong & undeferred : {targeted_metrics['wrong_and_undeferred_pct']:.2f}%")

# -------------------------------------------------------------
# The Gap
# -------------------------------------------------------------

gap_risk = (targeted_metrics['risk_on_predicted']
            - transferred_metrics['risk_on_predicted'])
gap_wrong_undeferred = (targeted_metrics['wrong_and_undeferred_pct']
                        - transferred_metrics['wrong_and_undeferred_pct'])

print("\n" + "=" * 70)
print("THE GAP")
print("=" * 70)
print(f"  Risk on predicted — transferred : {transferred_metrics['risk_on_predicted']:.2f}%")
print(f"  Risk on predicted — targeted    : {targeted_metrics['risk_on_predicted']:.2f}%")
print(f"  Δ risk                          : {gap_risk:+.2f} pts")
print(f"")
print(f"  Wrong-and-undeferred — transferred : {transferred_metrics['wrong_and_undeferred_pct']:.2f}%")
print(f"  Wrong-and-undeferred — targeted    : {targeted_metrics['wrong_and_undeferred_pct']:.2f}%")
print(f"  Δ wrong-and-undeferred             : {gap_wrong_undeferred:+.2f} pts")



Evaluation of Deep Ensembles under three conditions

CLEAN:
  coverage           : 76.30%
  deferral rate      : 23.70%
  risk on predicted  : 9.99%
  wrong & undeferred : 7.62%

TRANSFERRED PGD (baseline-attacked, transferred to ensemble):
  coverage           : 70.80%
  deferral rate      : 29.20%
  risk on predicted  : 15.50%
  wrong & undeferred : 10.97%

Running targeted gate attack with lambda_agreement = 5.0 (slower — gradients through all members)...

TARGETED GATE ATTACK (misclassify + preserve agreement):
  coverage           : 93.75%
  deferral rate      : 6.25%
  risk on predicted  : 45.73%
  wrong & undeferred : 42.88%

THE GAP
  Risk on predicted — transferred : 15.50%
  Risk on predicted — targeted    : 45.73%
  Δ risk                          : +30.23 pts

  Wrong-and-undeferred — transferred : 10.97%
  Wrong-and-undeferred — targeted    : 42.88%
  Δ wrong-and-undeferred             : +31.90 pts


In [ ]:

# -------------------------------------------------------------
# Part H: Head-to-Head with Essay #7
# -------------------------------------------------------------
#
# NOTE: these Essay #7 values are reference results from a different
# notebook run, not re-evaluated here -- consistent with how #6-#8
# treated cross-essay comparisons. Check whether "This essay —
# transferred attack" is close to 15.42% before treating this table
# as a controlled comparison; if it diverges substantially, only the
# WITHIN-ESSAY gap (targeted vs. this essay's transferred) should be
# read as conclusive.
# -------------------------------------------------------------

essay7_reference = {
    'Deep Ensembles (transferred, Essay #7)': {
        'adv_risk': 15.42,
        'adv_deferral': 26.39,
        'source': 'Essay #7',
    },
}

print("\n" + "=" * 70)
print("Head-to-Head with Essay #7")
print("=" * 70)
print(f"{'Method':<45} {'Adv Risk':<12} {'Deferral':<12} {'Source':<12}")
print("-" * 85)
for name, r in essay7_reference.items():
    print(f"{name:<45} {r['adv_risk']:>6.2f}%      {r['adv_deferral']:>6.2f}%      {r['source']:<12}")

print(f"{'This essay — transferred attack':<45} "
      f"{transferred_metrics['risk_on_predicted']:>6.2f}%      "
      f"{transferred_metrics['deferral_rate']:>6.2f}%      This essay")

print(f"{'This essay — targeted gate attack':<45} "
      f"{targeted_metrics['risk_on_predicted']:>6.2f}%      "
      f"{targeted_metrics['deferral_rate']:>6.2f}%      This essay")

divergence = abs(transferred_metrics['risk_on_predicted'] - essay7_reference['Deep Ensembles (transferred, Essay #7)']['adv_risk'])
if divergence > 3.0:
    print(f"\nNOTE: this essay's transferred-attack risk diverges from Essay #7's "
          f"reference by {divergence:.2f} points. Treat the cross-essay comparison "
          f"above as contextual, not controlled -- the within-essay gap (targeted "
          f"vs. this essay's own transferred attack) is still valid.")



Head-to-Head with Essay #7
Method                                        Adv Risk     Deferral     Source      
-------------------------------------------------------------------------------------
Deep Ensembles (transferred, Essay #7)         15.42%       26.39%      Essay #7    
This essay — transferred attack                15.50%       29.20%      This essay
This essay — targeted gate attack              45.73%        6.25%      This essay


In [ ]:

# -------------------------------------------------------------
# Part I: Summary and Outputs
# -------------------------------------------------------------

summary = {
    'clean_accuracy': clean_acc,
    'threshold': THRESHOLD,
    'calibration_deferral': cal_deferral,
    'lambda_sweep': {str(k): v for k, v in sweep_results.items()},
    'selected_lambda': BEST_LAMBDA,
    'dry_run_trace': trace,
    'clean': clean_metrics,
    'transferred': transferred_metrics,
    'targeted': targeted_metrics,
    'gap_risk': gap_risk,
    'gap_wrong_undeferred': gap_wrong_undeferred,
}

with open('essay9_targeted_gate_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n" + "=" * 55)
print("Experiment Summary")
print("=" * 55)
print(f"""
Deep Ensemble clean accuracy : {clean_acc:.2f}%
Gate threshold               : {THRESHOLD:.6f}
Calibration deferral         : {cal_deferral*100:.2f}%
Selected lambda_agreement    : {BEST_LAMBDA}

Clean:
  risk on predicted          : {clean_metrics['risk_on_predicted']:.2f}%
  wrong & undeferred         : {clean_metrics['wrong_and_undeferred_pct']:.2f}%

Transferred PGD:
  risk on predicted          : {transferred_metrics['risk_on_predicted']:.2f}%
  wrong & undeferred         : {transferred_metrics['wrong_and_undeferred_pct']:.2f}%

Targeted gate attack:
  risk on predicted          : {targeted_metrics['risk_on_predicted']:.2f}%
  wrong & undeferred         : {targeted_metrics['wrong_and_undeferred_pct']:.2f}%

The Gap:
  Δ risk on predicted        : {gap_risk:+.2f} pts
  Δ wrong & undeferred       : {gap_wrong_undeferred:+.2f} pts

Output files:
  - checkpoint_baseline_essay9.pth
  - essay9_targeted_gate_results.json
""")

print("\nNotebook complete.")



Experiment Summary

Deep Ensemble clean accuracy : 84.90%
Gate threshold               : 0.054999
Calibration deferral         : 25.00%
Selected lambda_agreement    : 5.0

Clean:
  risk on predicted          : 9.99%
  wrong & undeferred         : 7.62%

Transferred PGD:
  risk on predicted          : 15.50%
  wrong & undeferred         : 10.97%

Targeted gate attack:
  risk on predicted          : 45.73%
  wrong & undeferred         : 42.88%

The Gap:
  Δ risk on predicted        : +30.23 pts
  Δ wrong & undeferred       : +31.90 pts

Output files:
  - checkpoint_baseline_essay9.pth
  - essay9_targeted_gate_results.json


Notebook complete.
